#### Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import os
import re
import random
import numpy as np
from glob import glob
import cv2

#### Custom Dataset Class for Writer Verification (Image Pairs)

In [ ]:
class WriterVerificationDataset(Dataset):
    def __init__(self, image_paths, num_pairs=1000, img_h=224, img_w=224, transform=None):
        """
        Args:
            root_dir (str): Path to 'training_images' directory.
            num_pairs (int): Number of total pairs to generate.
            img_h, img_w (int): Image height and width to resize to.
            transform (callable, optional): Optional transform to apply to images.
        """
        self.image_paths = image_paths
        self.img_h = img_h
        self.img_w = img_w
        self.transform = transform
        self.writer_dict = self._build_writer_dict()
        self.writer_ids = list(self.writer_dict.keys())
        self.num_pairs = num_pairs

        # Pre-generate pair info for deterministic access
        self.pairs = self._generate_pairs()


    def _build_writer_dict(self):
        writer_dict = {}
        id_pattern = re.compile(r'(id_[A-Za-z0-9]+)')

        for img_path in self.image_paths:
            match = id_pattern.search(os.path.basename(img_path))
            if match:
                writer_id = match.group(1)
                writer_dict.setdefault(writer_id, []).append(img_path)

        return writer_dict
    

    def _generate_pairs(self):
        pairs = []
        for _ in range(self.num_pairs):
            if random.random() < 0.5:
                # Positive pair (same writer)
                writer = random.choice(self.writer_ids)
                if len(self.writer_dict[writer]) >= 2:
                    img1, img2 = random.sample(self.writer_dict[writer], 2)
                    label = 1
                else:
                    continue  # skip this one
            else:
                # Negative pair (different writers)
                w1, w2 = random.sample(self.writer_ids, 2)
                img1 = random.choice(self.writer_dict[w1])
                img2 = random.choice(self.writer_dict[w2])
                label = 0
                
            pairs.append((img1, img2, label))
        return pairs


    def __len__(self):
        return len(self.pairs)


    def _load_image(self, path):

        # PREPROCESSING TO BE EDITED
        img = cv2.imread(path, 0)  # grayscale
        img = cv2.resize(img, (self.img_w, self.img_h))
        img = img.astype(np.float32) / 255.0  # normalize to [0, 1]
        img = np.expand_dims(img, axis=0)  # shape: [1, H, W]

        
        if self.transform:
            img = self.transform(img)
        return torch.tensor(img)


    def __getitem__(self, idx):
        img1_path, img2_path, label = self.pairs[idx]
        img1 = self._load_image(img1_path)
        img2 = self._load_image(img2_path)
        label = torch.tensor([label], dtype=torch.float32)
        return (img1, img2), label

In [ ]:
'''
class VerificationDataset(Dataset):
    def __init__(self, root_dir, transform=None, num_pairs=1000):
        """
        root_dir: Directory with subfolders for each writer containing images.
        transform: Image transformations (e.g., resizing, normalization).
        num_pairs: Number of image pairs to generate.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.writer_images = {}
        self.pairs = []
        self.labels = []

        # Load images per writer
        writer_folders = sorted(os.listdir(root_dir))
        for writer in writer_folders:
            writer_path = os.path.join(root_dir, writer)
            self.writer_images[writer] = [os.path.join(writer_path, img) for img in os.listdir(writer_path)]

        # Generate image pairs (same and different writers)
        np.random.seed(42)
        for _ in range(num_pairs // 2):  # Half same, half different
            # Same writer pair
            writer = np.random.choice(writer_folders)
            if len(self.writer_images[writer]) >= 2:
                img1, img2 = np.random.choice(self.writer_images[writer], size=2, replace=False)
                self.pairs.append((img1, img2))
                self.labels.append(1)  # Same writer

            # Different writer pair
            writer1, writer2 = np.random.choice(writer_folders, size=2, replace=False)
            img1 = np.random.choice(self.writer_images[writer1])
            img2 = np.random.choice(self.writer_images[writer2])
            self.pairs.append((img1, img2))
            self.labels.append(0)  # Different writer

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_path, img2_path = self.pairs[idx]
        label = self.labels[idx]
        img1 = Image.open(img1_path).convert('RGB')
        img2 = Image.open(img2_path).convert('RGB')

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, label
'''

#### NetVLAD Layer

In [ ]:
# NetVLAD Layer with Dropout
'''
class NetVLAD(nn.Module):
    def __init__(self, num_clusters=32, dim=512, normalize_input=True):
        super(NetVLAD, self).__init__()
        self.num_clusters = num_clusters
        self.dim = dim
        self.normalize_input = normalize_input
        self.conv = nn.Conv2d(dim, num_clusters, kernel_size=1, bias=True)
        self.centroids = nn.Parameter(torch.randn(num_clusters, dim))
        self.dropout = nn.Dropout(p=0.3)  # Added for regularization
        self._init_params()
    def _init_params(self):
        nn.init.normal_(self.conv.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.conv.bias, 0.0)
    def forward(self, x):
        batch_size, dim, H, W = x.size()
        if self.normalize_input:
            x = nn.functional.normalize(x, p=2, dim=1)
        soft_assign = self.conv(x).view(batch_size, self.num_clusters, H * W)
        soft_assign = nn.functional.softmax(soft_assign, dim=1)
        x_flatten = x.view(batch_size, dim, H * W)
        residual = x_flatten.expand(self.num_clusters, -1, -1, -1).permute(1, 0, 2, 3) - \
                   self.centroids.expand(H * W, -1, -1).permute(1, 2, 0).unsqueeze(0)
        residual *= soft_assign.unsqueeze(2)
        vlad = residual.sum(dim=-1)
        vlad = nn.functional.normalize(vlad, p=2, dim=2)
        vlad = vlad.view(batch_size, -1)
        vlad = nn.functional.normalize(vlad, p=2, dim=1)
        vlad = self.dropout(vlad)  # Apply dropout
        return vlad
'''

In [ ]:
class NetVLAD(nn.Module):
    def __init__(self, num_clusters=64, dim=512, normalize_input=True):
        super(NetVLAD, self).__init__()
        self.num_clusters = num_clusters
        self.dim = dim
        self.normalize_input = normalize_input
        self.conv = nn.Conv2d(dim, num_clusters, kernel_size=1, bias=True)
        self.centroids = nn.Parameter(torch.randn(num_clusters, dim))
        self._init_params()

    def _init_params(self):
        nn.init.normal_(self.conv.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.conv.bias, 0.0)

    def forward(self, x):
        batch_size, dim, H, W = x.size()
        if self.normalize_input:
            x = nn.functional.normalize(x, p=2, dim=1)
        soft_assign = self.conv(x).view(batch_size, self.num_clusters, H * W)
        soft_assign = nn.functional.softmax(soft_assign, dim=1)
        x_flatten = x.view(batch_size, dim, H * W)
        residual = x_flatten.expand(self.num_clusters, -1, -1, -1).permute(1, 0, 2, 3) - \
                   self.centroids.expand(H * W, -1, -1).permute(1, 2, 0).unsqueeze(0)
        residual *= soft_assign.unsqueeze(2)
        vlad = residual.sum(dim=-1)
        vlad = nn.functional.normalize(vlad, p=2, dim=2)
        vlad = vlad.view(batch_size, -1)
        vlad = nn.functional.normalize(vlad, p=2, dim=1)
        return vlad

#### Siamese Network for Writer Verification

In [ ]:
class WriterVerificationModel(nn.Module):
    def __init__(self, num_clusters=64):
        super(WriterVerificationModel, self).__init__()
        # Shared ResNet-50 backbone
        self.resnet = models.resnet50(pretrained=True)
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-2])
        self.netvlad = NetVLAD(num_clusters=num_clusters, dim=2048)

    def forward(self, x1, x2):
        # Extract features for both images
        feat1 = self.resnet(x1)
        feat1 = self.netvlad(feat1)
        feat2 = self.resnet(x2)
        feat2 = self.netvlad(feat2)
        return feat1, feat2

#### Contrastive Loss

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, feat1, feat2, label):
        # Compute Euclidean distance between feature vectors
        dist = torch.nn.functional.pairwise_distance(feat1, feat2)
        # Contrastive loss
        loss_same = label * torch.pow(dist, 2)
        loss_diff = (1 - label) * torch.pow(torch.clamp(self.margin - dist, min=0.0), 2)
        loss = 0.5 * (loss_same + loss_diff)
        return loss.mean()

In [ ]:
class CosineContrastiveLoss(nn.Module):
    def __init__(self, margin=0.5):
        super(CosineContrastiveLoss, self).__init__()
        self.margin = margin  # Margin for cosine distance (0 to 2)

    def forward(self, feat1, feat2, label):
        # Compute cosine similarity
        cosine_sim = nn.functional.cosine_similarity(feat1, feat2, dim=1)
        # Cosine distance = 1 - cosine similarity
        cosine_dist = 1 - cosine_sim
        
        # Contrastive loss
        loss_same = label * cosine_dist  # Minimize distance for same writer
        loss_diff = (1 - label) * torch.clamp(self.margin - cosine_dist, min=0.0)  # Maximize distance up to margin
        loss = 0.5 * (loss_same + loss_diff)
        return loss.mean()

#### Training Function


In [ ]:
# Training Function with Early Stopping
'''
def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=5, is_finetune=False):
    model.train()
    if is_finetune:
        # Freeze layers up to layer3
        for name, param in model.resnet.named_parameters():
            if 'layer4' not in name:  # Freeze conv1, layer1, layer2, layer3
                param.requires_grad = False
    best_val_acc = 0
    patience = 3
    epochs_no_improve = 0
    for epoch in range(num_epochs):
        running_loss = 0.0
        for img1, img2, labels in train_loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.float().to(device)
            optimizer.zero_grad()
            feat1, feat2 = model(img1, img2)
            loss = criterion(feat1, feat2, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")
        # Validate
        val_acc = evaluate_model(model, val_loader, device, threshold=0.2)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), 'best_model.pt')  # Save best model
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping triggered")
            break
    if is_finetune:
        model.load_state_dict(torch.load('best_model.pt'))  # Load best model
'''

In [ ]:
def train_model(model, train_loader, criterion, optimizer, device, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for img1, img2, labels in train_loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.float().to(device)
            
            optimizer.zero_grad()
            feat1, feat2 = model(img1, img2)
            loss = criterion(feat1, feat2, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

#### Evaluation Function

In [ ]:
def evaluate_model(model, test_loader, device, threshold=0.5):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for img1, img2, labels in test_loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
            feat1, feat2 = model(img1, img2)
            dist = torch.nn.functional.pairwise_distance(feat1, feat2)
            predictions = (dist < threshold).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    
    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [ ]:
'''
def evaluate_model(model, test_loader, device, threshold=0.2):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for img1, img2, labels in test_loader:
            img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
            feat1, feat2 = model(img1, img2)
            cosine_sim = nn.functional.cosine_similarity(feat1, feat2, dim=1)
            cosine_dist = 1 - cosine_sim
            predictions = (cosine_dist < threshold).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    accuracy = 100 * correct / total
    print(f"Accuracy: {accuracy:.2f}%")
    return accuracy
'''

#### Main Execution


In [ ]:
def main():
    # Hyperparameters
    batch_size = 32
    num_epochs = 10
    learning_rate = 0.001
    num_clusters = 64
    margin = 1.0  # Contrastive loss margin
    threshold = 0.5  # Similarity threshold for evaluation

    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Data transformations
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Load dataset (replace 'path/to/dataset' with actual path)
    # Training: num_pairs = 120
    # Validation: num_pairs = 20
    # Testing: num_pairs = 20

    # dataset = WriterVerificationDataset(full_train_data, num_pairs=500)
    train_dataset = VerificationDataset(root_dir='path/to/train_dataset', transform=transform, num_pairs=1000)
    test_dataset = VerificationDataset(root_dir='path/to/test_dataset', transform=transform, num_pairs=200)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model, loss, and optimizer
    model = WriterVerificationModel(num_clusters=num_clusters).to(device)
    criterion = ContrastiveLoss(margin=margin)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Train the model
    train_model(model, train_loader, criterion, optimizer, device, num_epochs)

    # Evaluate the model
    evaluate_model(model, test_loader, device, threshold)

In [ ]:
# Main Execution
def main():
    # Hyperparameters
    batch_size = 16
    num_epochs = 10
    learning_rate = 0.0001
    num_clusters = 32
    margin = 0.5
    threshold = 0.2

    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Data transformations with augmentation
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(10),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Split writers (example: manually assign to avoid overlap)
    all_writers = sorted(os.listdir('path/to/dataset'))
    np.random.seed(42)
    np.random.shuffle(all_writers)
    train_writers = all_writers[:50]  # ~80% (198 images, ~45-50 writers)
    val_writers = all_writers[50:55]  # ~10% (25 images, ~5-7 writers)
    test_writers = all_writers[55:60]  # ~10% (24 images, ~5-7 writers)

    # Load datasets
    train_dataset = VerificationDataset(root_dir='path/to/dataset', transform=transform, num_pairs=120, writers=train_writers)
    val_dataset = VerificationDataset(root_dir='path/to/dataset', transform=transform, num_pairs=20, writers=val_writers)
    test_dataset = VerificationDataset(root_dir='path/to/dataset', transform=transform, num_pairs=20, writers=test_writers)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model, loss, and optimizer
    model = WriterVerificationModel(num_clusters=num_clusters).to(device)
    criterion = CosineContrastiveLoss(margin=margin)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Train and evaluate
    train_model(model, train_loader, criterion, optimizer, device, num_epochs)
    evaluate_model(model, val_loader, device, threshold)  # Validate
    evaluate_model(model, test_loader, device, threshold)  # Test


In [ ]:
'''
def main():
    # Hyperparameters
    batch_size = 16
    pretrain_epochs = 10
    finetune_epochs = 5
    pretrain_lr = 0.001
    finetune_lr = 1e-5  # Low learning rate for fine-tuning
    num_clusters = 32
    margin = 0.5
    threshold = 0.2
    weight_decay = 1e-4  # Regularization

    # Device configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Data transformations
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(10),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Pretraining dataset (e.g., KHATT)
    pretrain_dataset = VerificationDataset(
        root_dir='path/to/khatt_dataset',
        transform=transform,
        num_pairs=10000,
        writers=None
    )
    pretrain_loader = DataLoader(pretrain_dataset, batch_size=batch_size, shuffle=True)

    # Your dataset: Split writers
    all_writers = sorted(os.listdir('path/to/your_dataset'))
    np.random.seed(42)
    np.random.shuffle(all_writers)
    train_writers = all_writers[:50]  # ~198 images
    val_writers = all_writers[50:55]  # ~25 images
    test_writers = all_writers[55:60]  # ~24 images

    # Your datasets
    train_dataset = VerificationDataset(
        root_dir='path/to/your_dataset',
        transform=transform,
        num_pairs=120,
        writers=train_writers
    )
    val_dataset = VerificationDataset(
        root_dir='path/to/your_dataset',
        transform=transform,
        num_pairs=20,
        writers=val_writers
    )
    test_dataset = VerificationDataset(
        root_dir='path/to/your_dataset',
        transform=transform,
        num_pairs=20,
        writers=test_writers
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model
    model = WriterVerificationModel(num_clusters=num_clusters).to(device)
    criterion = CosineContrastiveLoss(margin=margin)

    # Step 1: Pretrain on KHATT
    print("Pretraining on KHATT dataset...")
    optimizer = optim.Adam(model.parameters(), lr=pretrain_lr, weight_decay=weight_decay)
    train_model(model, pretrain_loader, val_loader, criterion, optimizer, device, num_epochs=pretrain_epochs, is_finetune=False)

    # Step 2: Fine-tune on your dataset
    print("Fine-tuning on your dataset...")
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=finetune_lr, weight_decay=weight_decay)
    train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=finetune_epochs, is_finetune=True)

    # Evaluate
    print("Evaluating on test set...")
    evaluate_model(model, test_loader, device, threshold)
'''

In [ ]:
if __name__ == "__main__":
    main()